# pyrepl

> A Python prompt you own, and an agent in the layer above it.

One terminal holds two things. The Python prompt's namespace belongs to whoever is typing;
the agent reads that namespace and builds in a layer of its own, and cannot rebind a name it
did not create. That guarantee is not this module's: Dhrishti serves the kernel's namespace
and splits its API in two, and everything here does is point the agent at the half that
cannot mutate anything.

In [ ]:
#| default_exp pyrepl

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import test_eq, test_fail

In [ ]:
#| export
import asyncio, codeop, json, os, queue, shutil, sys, tempfile, threading, urllib.parse, urllib.request
from dataclasses import dataclass, field
from pathlib import Path
from rich.text import Text
from ramabana.core import agent_err
from ramabana.tools import LocalHost, WRITE_TOOLS
from ramabana.agent import Agent, Approvals
from ramabana.cli import Ui, GRUVBOX, HELP

## Outputs

A kernel reports its results as a stream of messages; a notebook stores them as a list of
dicts. `ExecOutcome` is one request in the notebook's shape, which is what lets the same
outputs go to the terminal, to `log_cell` and to a test without a second representation.

In [ ]:
#| export
@dataclass
class ExecOutcome:
    "One kernel request in nbformat's output shape."
    ok: bool = True
    outputs: list = field(default_factory=list)
    execution_count: int | None = None
    error: str | None = None

def output_text(outputs):
    "Flatten notebook outputs for tests, logs and plain terminal fallbacks."
    parts = []
    for out in outputs:
        kind = out.get('output_type')
        if kind == 'stream': parts.append(_text(out.get('text')))
        elif kind in ('execute_result', 'display_data'):
            data = out.get('data') or {}
            parts.append(_text(data.get('text/plain') or data.get('text/markdown')))
        elif kind == 'error':
            trace = out.get('traceback') or []
            parts.append('\n'.join(trace) if trace else f"{out.get('ename')}: {out.get('evalue')}")
    return '\n'.join(p.rstrip('\n') for p in parts if p is not None)

def _text(value):
    return ''.join(value) if isinstance(value, list) else str(value or '')

In [ ]:
outs = [{'output_type': 'stream', 'text': 'hello\n'},
        {'output_type': 'execute_result', 'data': {'text/plain': '42'}},
        {'output_type': 'error', 'ename': 'ValueError', 'evalue': 'bad', 'traceback': []}]
test_eq(output_text(outs), 'hello\n42\nValueError: bad')
# A stream arrives split across messages, so the list form has to flatten rather than repr.
test_eq(output_text([{'output_type': 'stream', 'text': ['a', 'b']}]), 'ab')
# An error with a traceback prefers it: the ename alone loses where it happened.
test_eq(output_text([{'output_type': 'error', 'ename': 'E', 'evalue': 'v',
                      'traceback': ['line one', 'line two']}]), 'line one\nline two')
test_eq(output_text([]), '')

## The kernel

A private `ipykernel`, and Dhrishti started *inside* it — because the namespace Dhrishti
serves is the kernel's, and because its server runs on a background thread, which is what
keeps it answering during exactly the long cell you most want to watch.

The port comes back through a printed marker rather than a return value: the bootstrap runs
as a cell, and a cell's only channel to the caller is its output.

In [ ]:
#| export
class Kernel:
    "A private ipykernel with Dhrishti serving its live namespace."
    def __init__(self, cwd='.'):
        self.cwd = Path(cwd).resolve()
        self.km = self.kc = None
        self.base = None
        self._ipc_dir = None
        self._exec_lock = asyncio.Lock()
        self._shell_lock = asyncio.Lock()

    @property
    def alive(self):
        return self.km is not None and self.kc is not None and self.km.has_kernel

    async def start(self, timeout=60):
        from jupyter_client.kernelspec import KernelSpec
        from jupyter_client.manager import AsyncKernelManager
        opts = {'kernel_name': 'python3'}
        if os.name != 'nt':
            self._ipc_dir = tempfile.mkdtemp(prefix='rama-k-', dir='/tmp' if os.path.isdir('/tmp') else None)
            opts.update(transport='ipc', ip=os.path.join(self._ipc_dir, 'k'))
        self.km = AsyncKernelManager(**opts)
        self.km._kernel_spec = KernelSpec(
            argv=[sys.executable, '-m', 'ipykernel_launcher', '-f', '{connection_file}'],
            display_name='Ramabana PyREPL', language='python')
        try:
            await self.km.start_kernel(cwd=str(self.cwd))
            self.kc = self.km.client()
            self.kc.start_channels()
            await self.kc.wait_for_ready(timeout=timeout)
            await self._bootstrap()
            return self
        except BaseException:
            await self.shutdown()
            raise

    async def _bootstrap(self):
        root = self.cwd/'.ramabana'/'pyrepl'
        source = (
            'def _ramabana_bootstrap():\n'
            ' import dhrishti.serving as ds\n'
            f' p = ds.serve_in_kernel(name="ramabana pyrepl", agent="restricted", token=True, '
            f'session_dir={str(root/"sessions")!r}, agent_session_dir={str(root/"agents")!r})\n'
            ' print("__RAMABANA_DHRISHTI__" + str(p))\n'
            '_ramabana_bootstrap()\n'
            'del _ramabana_bootstrap')
        result = await self.execute(source, store_history=False)
        marker = next((line for out in result.outputs if out.get('output_type') == 'stream'
                       for line in _text(out.get('text')).splitlines()
                       if line.startswith('__RAMABANA_DHRISHTI__')), '')
        if not result.ok or not marker:
            raise RuntimeError(result.error or output_text(result.outputs) or 'Dhrishti did not start')
        self.base = f'http://127.0.0.1:{int(marker.removeprefix("__RAMABANA_DHRISHTI__"))}'

    async def execute(self, code, store_history=True, on_output=None):
        "Execute one cell and stream each nbformat-shaped output to `on_output`."
        if not self.alive: return ExecOutcome(ok=False, error='kernel is not running')
        async with self._exec_lock, self._shell_lock:
            msg_id = self.kc.execute(str(code), store_history=store_history, allow_stdin=False)
            outputs = await self._collect(msg_id, on_output)
            reply = await self._reply(msg_id)
        content = (reply or {}).get('content') or {}
        error = None
        if content.get('status') == 'error': error = f"{content.get('ename')}: {content.get('evalue')}"
        elif content.get('status') == 'abort': error = 'aborted'
        if error is None:
            err = next((o for o in outputs if o.get('output_type') == 'error'), None)
            if err: error = f"{err.get('ename')}: {err.get('evalue')}"
        return ExecOutcome(error is None, outputs, content.get('execution_count'), error)

    async def _collect(self, msg_id, on_output):
        outputs, displays = [], {}
        while True:
            try: message = await self.kc.get_iopub_msg(timeout=.5)
            except (queue.Empty, asyncio.TimeoutError):
                if not self.alive: break
                continue
            if (message.get('parent_header') or {}).get('msg_id') != msg_id: continue
            kind, content = message['header']['msg_type'], message['content']
            if kind == 'status' and content.get('execution_state') == 'idle': break
            if kind == 'clear_output': outputs.clear(); displays.clear(); continue
            out = self._output(kind, content)
            if out is None: continue
            display_id = (content.get('transient') or {}).get('display_id')
            if kind == 'update_display_data' and display_id in displays:
                outputs[displays[display_id]] = out
            elif (out['output_type'] == 'stream' and outputs and outputs[-1].get('output_type') == 'stream'
                  and outputs[-1].get('name') == out.get('name')):
                outputs[-1]['text'] += out['text']
            else:
                outputs.append(out)
                if display_id: displays[display_id] = len(outputs) - 1
            if on_output: on_output(out)
        return outputs

    @staticmethod
    def _output(kind, content):
        if kind == 'stream':
            return {'output_type': 'stream', 'name': content.get('name', 'stdout'), 'text': content.get('text', '')}
        if kind == 'error':
            return {'output_type': 'error', 'ename': content.get('ename', ''),
                    'evalue': content.get('evalue', ''), 'traceback': list(content.get('traceback') or [])}
        if kind in ('execute_result', 'display_data', 'update_display_data'):
            return {'output_type': 'execute_result' if kind == 'execute_result' else 'display_data',
                    'data': content.get('data') or {}, 'metadata': content.get('metadata') or {}}
        return None

    async def _reply(self, msg_id):
        while True:
            try: message = await self.kc.get_shell_msg(timeout=10)
            except (queue.Empty, asyncio.TimeoutError): return None
            if (message.get('parent_header') or {}).get('msg_id') == msg_id: return message

    async def complete(self, code, pos):
        async with self._shell_lock:
            msg_id = self.kc.complete(str(code), int(pos))
            while True:
                message = await self.kc.get_shell_msg(timeout=10)
                if (message.get('parent_header') or {}).get('msg_id') == msg_id:
                    content = message.get('content') or {}
                    return content.get('matches') or [], int(content.get('cursor_start', pos))

    async def interrupt(self):
        if self.km: await self.km.interrupt_kernel()

    async def shutdown(self):
        if self.kc:
            try: self.kc.stop_channels()
            except Exception: pass
        if self.km and self.km.has_kernel:
            try: await self.km.shutdown_kernel(now=False)
            except Exception: pass
        self.km = self.kc = self.base = None
        if self._ipc_dir:
            shutil.rmtree(self._ipc_dir, ignore_errors=True)
            self._ipc_dir = None

In [ ]:
kernel = await Kernel('.').start()
r = await kernel.execute('a = 6 * 7\na')
test_eq((r.ok, output_text(r.outputs)), (True, '42'))
assert kernel.base.startswith('http://127.0.0.1:')     # dhrishti came up inside the kernel

In [ ]:
# A dead kernel answers rather than hanging on a channel nobody is writing to.
dead = Kernel('.')
r = await dead.execute('1 + 1')
test_eq((r.ok, r.error), (False, 'kernel is not running'))

# An error is an outcome, not an exception: the caller is a UI.
r = await kernel.execute('1/0')
test_eq(r.ok, False)
assert 'ZeroDivisionError' in r.error and 'ZeroDivisionError' in output_text(r.outputs)

# Streams are coalesced, so a loop printing forty lines is one output.
r = await kernel.execute("for i in range(3): print(i)")
test_eq(len([o for o in r.outputs if o['output_type'] == 'stream']), 1)
test_eq(output_text(r.outputs), '0\n1\n2')

# Completion comes from the kernel, which is IPython's completer with jedi behind it.
await kernel.execute('import json')
matches, start = await kernel.complete('json.du', 7)
assert 'json.dump' in matches or 'dump' in matches

## The host

An ordinary Ramabana host — the file, search and web tools are the same ones — whose *Python*
tools go over HTTP to the agent half of the Dhrishti API. That half is the ungated one, and
that is the whole protection: the token that opens `/api/exec`, `/api/set` and `/api/promote`
is never read here, so there is no code path from a tool call to the owner's namespace.

`log_cell` is what makes the session one artifact. The owner's cells go in as code with their
real outputs, each turn as `**user**`/`**assistant**` markdown, in the order they happened.

In [ ]:
#| export
def _api(base, path, params=None, timeout=60):
    query = urllib.parse.urlencode(params or {})
    url = base.rstrip('/') + path + (('?' + query) if query else '')
    with urllib.request.urlopen(url, timeout=timeout) as response:
        return json.loads(response.read())

class DhrishtiHost(LocalHost):
    "A project host whose Python tools use a protected Dhrishti overlay."
    def __init__(self, roots, base, **kwargs):
        super().__init__(roots, **kwargs)
        self.base = base
        self.agent_log = Path((_api(base, '/agent/api/info') or {}).get('log') or '')

    def run_python(self, code):
        # `scope='overlay'`, always. The agent gets the sandboxed layer whatever it asks for,
        # because a scope is a tool argument and a tool argument is a thing a model can set.
        try: result = _api(self.base, '/agent/api/exec', {'code': str(code), 'scope': 'overlay'})
        except Exception as exc: return agent_err(exc)
        if result.get('error'): return str(result['error'])
        out = result.get('stdout') or ''
        value = result.get('result')
        if isinstance(value, dict): value = value.get('value')
        if value is not None: out += ('\n' if out else '') + str(value)
        return out or '(ok)'

    def inspect_python(self, code, scope='isolated'):
        if scope not in self.scopes: return f'this host only honours {self.scopes}'
        try: result = _api(self.base, '/agent/api/exec', {'code': str(code), 'scope': scope})
        except Exception as exc: return agent_err(exc)
        if result.get('error'): return str(result['error'])
        out = result.get('stdout') or ''
        value = result.get('result')
        if isinstance(value, dict): value = value.get('value')
        if value is not None: out += ('\n' if out else '') + str(value)
        return out or '(ok)'

    @property
    def scopes(self): return ('isolated', 'overlay')

    @property
    def kernel_kind(self): return 'ipykernel'

    def list_vars(self):
        try: result = _api(self.base, '/agent/api/rows', {'profile': 'minimal', 'sort': 'name'})
        except Exception as exc: return agent_err(exc)
        return '\n'.join(f"{node.get('name')}: {node.get('type')} = {node.get('value')}"
                         for group in result.get('groups', []) for node in group.get('nodes', []))

    def log_cell(self, source, outputs=None, cell_type='code'):
        "Append a human Python or model markdown turn to Dhrishti's session notebook."
        # `read`/append/`write` per cell rather than a held handle: the owner's kernel writes
        # this file too, and a session that survives a crash is worth more than the syscalls.
        if not str(self.agent_log): return
        from fastcore.nbio import read_nb, write_nb, new_nb, mk_cell
        self.agent_log.parent.mkdir(parents=True, exist_ok=True)
        nb = read_nb(self.agent_log) if self.agent_log.exists() else new_nb([])
        cell = mk_cell(str(source), cell_type)
        if cell_type == 'code':
            cell['outputs'], cell['execution_count'] = list(outputs or []), None
        nb.cells.append(cell)
        write_nb(nb, self.agent_log)

In [ ]:
host = DhrishtiHost(['.'], kernel.base, web=False, index=False)
await kernel.execute("owner = {'kept': 1}")

test_eq(host.run_python('mine = len(owner)'), '(ok)')      # reads the owner freely
assert 'mine' in host.list_vars() and 'owner' in host.list_vars()

# The layering, from this side of it: the write lands in the agent's layer and the owner's
# binding is untouched. Dhrishti refuses it; this only has to not ask for a way round.
host.run_python('owner = None')
r = await kernel.execute('owner')
test_eq(output_text(r.outputs), "{'kept': 1}")

# A transport that is not there comes back as a sentence, because a tool cannot raise usefully.
# `__init__` fetches `/agent/api/info` eagerly and is ported verbatim, so a dead base fails
# there rather than inside `run_python`; the test catches at the same boundary a caller would.
try: unreachable = DhrishtiHost(['.'], 'http://127.0.0.1:1', web=False, index=False).run_python('1')
except Exception as exc: unreachable = agent_err(exc)
assert 'Error' in unreachable

In [ ]:
from fastcore.nbio import read_nb
r = await kernel.execute('logged = 1')
host.log_cell('logged = 1', r.outputs)
host.log_cell('**user**\n\nwhat is logged?', cell_type='markdown')
nb = read_nb(host.agent_log)
test_eq([c.cell_type for c in nb.cells[-2:]], ['code', 'markdown'])
test_eq(nb.cells[-2].outputs, r.outputs)          # the real outputs, not a rendering of them